In [111]:
import torch
import numpy as np
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from diffusers import DDPMPipeline, DDIMScheduler, DDPMScheduler
from tqdm.auto import tqdm
import time
from scipy import linalg
import matplotlib.pyplot as plt
from torch.nn.functional import adaptive_avg_pool2d
from torchvision.models import inception_v3
import os
from datetime import datetime

In [112]:
class InceptionModel:
    def __init__(self, device):
        self.device = device
        self.model = inception_v3(pretrained=True, transform_input=False).to(device)
        self.model.eval()
        self.model.fc = torch.nn.Identity()
    
    def get_features(self, images):
        images = torch.nn.functional.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
        
        with torch.no_grad():
            features = self.model(images)
        
        return features

In [113]:
def calculate_fid(real_features, generated_features):
    # Calculate mean and covariance for real features
    mu_real = np.mean(real_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    
    # Calculate mean and covariance for generated features
    mu_gen = np.mean(generated_features, axis=0)
    sigma_gen = np.cov(generated_features, rowvar=False)
    
    # Calculate the squared distance between means
    mu_diff = mu_real - mu_gen
    mu_dist = np.sum(mu_diff ** 2)
    
    # Calculate the matrix square root term
    sigma_real = sigma_real + np.eye(sigma_real.shape[0]) * 1e-6
    sigma_gen = sigma_gen + np.eye(sigma_gen.shape[0]) * 1e-6
    
    covmean = linalg.sqrtm(sigma_real @ sigma_gen)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    # Calculate FID
    fid = mu_dist + np.trace(sigma_real + sigma_gen - 2 * covmean)
    return fid

In [116]:
class DiffusionEvaluator:
    def __init__(self, model_id="google/ddpm-cifar10-32", device=None):
        self.model_id = model_id
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Initialize the diffusion model
        self.model = DDPMPipeline.from_pretrained(model_id)
        self.model.to(self.device)
        self.model.unet.eval()
        
        # Initialize the Inception model for FID calculation
        self.inception = InceptionModel(self.device)
        
        # Set up results directory
        self.results_dir = "diffusion_results"
        os.makedirs(self.results_dir, exist_ok=True)
        
        # Initialize results log
        self.log_file = os.path.join(self.results_dir, f"evaluation_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
        with open(self.log_file, 'w') as f:
            f.write(f"Diffusion Model Evaluation: {model_id}\n")
            f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            f.close()
    
    def log(self, message):
        print(message)
        with open(self.log_file, 'a') as f:
            f.write(message + "\n")
    
    def create_linear_schedule(self, num_steps=11):
        return np.linspace(999, 0, num_steps)
    
    def create_edm_schedule(self, num_steps=11):
        # EDM schedule parameters
        sigma_min = 0.002
        sigma_max = 80.0
        rho = 7.0  # Controls the shape of the curve
        
        # Generate the EDM sigmas
        steps = np.linspace(0, 1, num_steps)
        sigmas = sigma_min ** (1 / rho) + steps * (sigma_max ** (1 / rho) - sigma_min ** (1 / rho))
        sigmas = sigmas ** rho
        
        # Normalize to [0, 1000] range to match DDPM timesteps
        schedule = 999 * (sigmas - sigma_min) / (sigma_max - sigma_min)
        schedule = np.flip(schedule)
        schedule[-1] = 0
        
        return schedule
    
    def create_logsnr_schedule(self, num_steps=11):
        # LogSNR parameters
        min_logsnr = -20
        max_logsnr = 20
        
        # Generate log-SNR values
        logsnr_values = np.linspace(max_logsnr, min_logsnr, num_steps)
        
        # Convert to noise schedule (timesteps)
        alphas_cumprod = 1 / (1 + np.exp(-logsnr_values))
        schedule = 999 * (1 - alphas_cumprod)
        schedule[-1] = 0
        
        return schedule
    
    def create_cosine_schedule(self, num_steps=11):
        s = 0.008
        steps = np.linspace(0, 1, num_steps)
        
        # Apply cosine formula
        alphas_cumprod = np.cos(((steps + s) / (1 + s)) * np.pi / 2) ** 2
        
        # Convert to timesteps
        schedule = 999 * (1 - alphas_cumprod)
        schedule = np.flip(schedule)
        schedule[-1] = 0
        
        return schedule
    
    def load_ays_schedule(self, ays_res=None):
        if ays_res:
            ays_schedule = np.array([0, 135, 255, 347, 430, 533, 649, 754, 846, 942, 999])
            ays_schedule = np.flip(ays_schedule)
        else:
            # Use the AYS schedule from the paper
            ays_schedule = np.array([0, 24, 124, 233, 343, 455, 545, 645, 736, 850, 999])
            ays_schedule = np.flip(ays_schedule)

        return ays_schedule

    def sample_with_schedule(self, scheduler_name, steps, batch_size=16, num_images=1000, seed=42):
        # Set random seed for reproducibility
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # Create scheduler
        scheduler = DDIMScheduler(
            num_train_timesteps=1000,
            beta_start=0.0001,
            beta_end=0.02,
            beta_schedule="linear",
            clip_sample=True
        )
        
        # Set up the timestep schedule
        if scheduler_name == "linear":
            timesteps = self.create_linear_schedule(steps + 1)
        elif scheduler_name == "edm":
            timesteps = self.create_edm_schedule(steps + 1)
        elif scheduler_name == "logsnr":
            timesteps = self.create_logsnr_schedule(steps + 1)
        elif scheduler_name == "cosine":
            timesteps = self.create_cosine_schedule(steps + 1)
        elif scheduler_name == "ays":
            timesteps = self.load_ays_schedule()
            # Interpolate if necessary to match the desired steps
            if len(timesteps) != steps + 1:
                original_indices = np.linspace(0, 1, len(timesteps))
                new_indices = np.linspace(0, 1, steps + 1)
                timesteps = np.interp(new_indices, original_indices, timesteps)
        else:
            raise ValueError(f"Unknown scheduler: {scheduler_name}")
        
        # Convert to integer timesteps
        custom_timesteps = [int(t) for t in timesteps[:-1]]  # Remove the last step (t=0)
        scheduler.timesteps = torch.tensor(custom_timesteps, device=self.device)
        scheduler.set_timesteps(len(custom_timesteps))
        print(f"The customized timestamp for:", custom_timesteps)

        # Update model with scheduler
        self.model.scheduler = scheduler
        
        # Generate samples
        start_time = time.time()
        generated_images = []
        num_batches = num_images // batch_size
        generator = torch.Generator(device=self.device)
        
        self.log(f"Generating {num_images} images with {scheduler_name} scheduler using {steps} steps...")
        
        for i in tqdm(range(num_batches)):
            # Generate a batch of images
            noise = torch.randn(
                batch_size,
                self.model.unet.config.in_channels,
                self.model.unet.config.sample_size,
                self.model.unet.config.sample_size,
                device=self.device
            )
            x = noise

            # Manual denoising loop
            for t in custom_timesteps:
                t_tensor = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
                with torch.no_grad():
                    noise_pred = self.model.unet(x, t_tensor).sample
                    x = scheduler.step(
                        model_output=noise_pred,
                        timestep=t,
                        sample=x,
                        generator=generator.manual_seed(seed + i)
                    ).prev_sample

            batch = (x / 2 + 0.5).clamp(0, 1)
            batch_tensor = batch.cpu()
            generated_images.append(batch_tensor)
        
        # Combine all batches
        generated_images = torch.cat(generated_images, dim=0)
        end_time = time.time()
        
        time_taken = end_time - start_time
        self.log(f"Generation with {scheduler_name} scheduler took {time_taken:.2f} seconds")
        
        return generated_images, time_taken

    def extract_features(self, images):
        features = []
        batch_size = 32
        num_batches = len(images) // batch_size + (1 if len(images) % batch_size != 0 else 0)
        
        for i in tqdm(range(num_batches)):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, len(images))
            batch = images[start_idx:end_idx].to(self.device)
            
            with torch.no_grad():
                batch_features = self.inception.get_features(batch)
                features.append(batch_features.cpu().numpy())
        
        return np.concatenate(features, axis=0)
    
    def get_real_features(self, num_images=1000, seed=42):
        # Set random seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        # Load CIFAR-10 dataset
        transform = transforms.Compose([transforms.ToTensor()])
        dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
        
        # Randomly select images
        indices = torch.randperm(len(dataset))[:num_images]
        loader = torch.utils.data.DataLoader(
            torch.utils.data.Subset(dataset, indices),
            batch_size=32,
            shuffle=False
        )
        
        # Extract features
        real_images = []
        for images, _ in tqdm(loader, desc="Loading real images"):
            real_images.append(images)
        
        real_images = torch.cat(real_images, dim=0)
        real_features = self.extract_features(real_images)
        
        return real_features
    
    def evaluate_schedulers(self, nfe_values=[10, 20, 30, 50], num_images=1000):
        # Schedulers to evaluate
        schedulers = ["linear", "edm", "logsnr", "cosine", "ays"]
#         schedulers = ["logsnr", "ays"]
        
        # Get real image features (only need to do this once)
        self.log("Extracting features from real CIFAR-10 images...")
        real_features = self.get_real_features(num_images)
        
        # Results table
        results = {}
        
        for nfe in nfe_values:
            self.log(f"\n--- Evaluating with NFE={nfe} ---")
            results[nfe] = {}
            
            for scheduler in schedulers:
                # Generate images
                generated_images, time_taken = self.sample_with_schedule(
                    scheduler, nfe, num_images=num_images
                )
                
                # Extract features
                self.log(f"Extracting features from generated images ({scheduler}, NFE={nfe})...")
                generated_features = self.extract_features(generated_images)
                
                # Calculate FID
                print("FID:", generated_features)
                fid = calculate_fid(real_features, generated_features)
                self.log(f"{scheduler.upper()} FID (NFE={nfe}): {fid:.2f}")
                
                # Save results
                results[nfe][scheduler] = {
                    "fid": fid,
                    "time": time_taken
                }
                
                # Save sample images
                self.save_sample_images(generated_images, scheduler, nfe)
        
        # Create summary table
        self.create_summary_table(results)
        
        return results
    
    def save_sample_images(self, images, scheduler_name, nfe, num_samples=16):
        # Select a subset of images
        indices = np.random.choice(len(images), min(num_samples, len(images)), replace=False)
        samples = [images[i] for i in indices]
        
        # Create grid
        grid_size = int(np.ceil(np.sqrt(len(samples))))
        fig, axs = plt.subplots(grid_size, grid_size, figsize=(10, 10))
        
        for i, ax in enumerate(axs.flatten()):
            if i < len(samples):
                img = samples[i].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
            ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.results_dir, f"{scheduler_name}_nfe{nfe}_samples.png"))
        plt.close()
    
    def create_summary_table(self, results):
        # Header
        header = "Sampling method"
        schedulers = list(results[list(results.keys())[0]].keys())
        nfe_values = list(results.keys())
        
        # Create table rows
        rows = []
        for scheduler in schedulers:
            row = [scheduler.upper()]
            for nfe in nfe_values:
                row.append(f"{results[nfe][scheduler]['fid']:.2f}")
            rows.append(row)
        
        # Create table string
        table = f"{header:<20}"
        for nfe in nfe_values:
            table += f"NFE={nfe:<10}"
        table += "\n" + "-" * 60 + "\n"
        
        for row in rows:
            table += f"{row[0]:<20}"
            for i in range(1, len(row)):
                table += f"{row[i]:<10}"
            table += "\n"
        
        # Save table
        self.log("\nSummary of FID scores:")
        self.log(table)
        
        # Create a plot of FID scores
        plt.figure(figsize=(10, 6))
        for scheduler in schedulers:
            fid_values = [results[nfe][scheduler]['fid'] for nfe in nfe_values]
            plt.plot(nfe_values, fid_values, marker='o', label=scheduler.upper())
        
        plt.xlabel('Number of Function Evaluations (NFE)')
        plt.ylabel('FID Score (lower is better)')
        plt.title('FID Scores vs. NFE for Different Schedulers')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(self.results_dir, "fid_comparison.png"))
        plt.close()

In [117]:
evaluator = DiffusionEvaluator()

results = evaluator.evaluate_schedulers(nfe_values=[10], num_images=100)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /home/svu/e0556748/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /home/svu/e0556748/.cache/huggingface/hub/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Extracting features from real CIFAR-10 images...
Files already downloaded and verified


Loading real images:   0%|          | 0/4 [00:00<?, ?it/s]

...

FID: [[0.24747425 0.17141324 0.00251688 ... 0.06904569 0.08872072 0.64011264]
 [0.46075058 0.01386528 0.14185166 ... 0.19668286 0.24918911 0.19630665]
 [0.9318454  0.4450001  0.03363554 ... 0.12250349 0.24662855 1.0301148 ]
 ...
 [0.33997416 0.07724907 0.00252673 ... 0.14946264 0.7121885  0.63112944]
 [0.41458154 0.04550281 0.03240979 ... 0.14434619 0.12764028 0.7374618 ]
 [0.36545324 0.02921201 0.01957378 ... 0.23335072 0.01644026 0.770823  ]]
COSINE FID (NFE=10): 232.06
The customized timestamp for: [999, 850, 736, 645, 545, 455, 343, 233, 124, 24]
Generating 100 images with ays scheduler using 10 steps...


  0%|          | 0/6 [00:00<?, ?it/s]

Generation with ays scheduler took 0.81 seconds
Extracting features from generated images (ays, NFE=10)...


  0%|          | 0/3 [00:00<?, ?it/s]

FID: [[0.4387645  0.13079521 0.08853054 ... 0.3179055  0.1528278  0.41725463]
 [0.7489084  0.03818376 0.04308109 ... 0.1848004  0.14384362 0.13687623]
 [0.5576042  0.21296264 0.17264432 ... 0.2142568  0.14701913 0.29160118]
 ...
 [0.6181125  0.44073668 0.1278127  ... 0.0706017  0.5471047  0.6537814 ]
 [0.5862009  0.18096174 0.05156965 ... 0.17228852 0.3865159  1.014199  ]
 [0.5161232  0.02165495 0.0017365  ... 0.10777459 0.23163977 0.3351845 ]]
AYS FID (NFE=10): 158.02

Summary of FID scores:
Sampling method     NFE=10        
------------------------------------------------------------
LINEAR              158.32    
EDM                 300.82    
LOGSNR              388.08    
COSINE              232.06    
AYS                 158.02    



In [118]:
evaluator.create_summary_table(results)


Summary of FID scores:
Sampling method     NFE=10        
------------------------------------------------------------
LINEAR              158.32    
EDM                 300.82    
LOGSNR              388.08    
COSINE              232.06    
AYS                 158.02    

